# Planck NPIPE noise: download (optional) + visualize

Fetch Planck **NPIPE (PR4)** noise Monte-Carlo maps from the Planck Legacy
Archive into

`/rds/rds-lxu/flamingo/integrated_maps_synthetic/planck_noise/npipe/`

**Default scope:** one realisation (`mc_00200`) × detector-set splits `A` and
`B` for 100–857 GHz. Files already under `OUT_ROOT` are skipped. No local
copies from other users' directories.

**FILE_ID** (verified against the live PLA):

```
npipe6v20_noise_{freq}_{A|B}_mc_{NNNNN}.fits
```

with 5-digit realisation padding (`00200`). See `../noise_description.md`.


## 1. Configuration

In [ ]:
from pathlib import Path

FREQUENCIES = (100, 143, 217, 353, 545, 857)
REAL = 200
SPLITS = ("A", "B")
DO_DOWNLOAD = True

OUT_ROOT = Path("/rds/rds-lxu/flamingo/integrated_maps_synthetic") / "planck_noise" / "npipe"
FIG_DIR = Path("../figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"freqs   : {list(FREQUENCIES)}")
print(f"real    : mc_{REAL:05d}")
print(f"splits  : {SPLITS}")
print(f"download: {DO_DOWNLOAD}")
print(f"out     : {OUT_ROOT}")


## 2. Resolve local paths under `OUT_ROOT` only

In [ ]:
def npipe_filename(freq, real, subset=""):
    sub = f"_{subset}" if subset else ""
    return f"npipe6v20_noise_{freq}{sub}_mc_{real:05d}.fits"

def out_path(freq, real, subset=""):
    sub = subset if subset else "full"
    return OUT_ROOT / f"{freq}GHz" / sub / npipe_filename(freq, real, subset)

def find_local(freq, real, subset=""):
    p = out_path(freq, real, subset)
    return p if p.is_file() else None

inventory = {}
missing = []
for freq in FREQUENCIES:
    for subset in SPLITS:
        p = find_local(freq, REAL, subset)
        inventory[(freq, subset)] = p
        tag = f"{freq}GHz {subset} mc_{REAL:05d}"
        if p is None:
            missing.append((freq, subset))
            print(f"MISSING  {tag}")
        else:
            print(f"found    {tag}  <- {p}")

print(f"\n{len(missing)} missing of {len(FREQUENCIES)*len(SPLITS)}")
print("example URL:",
      "https://pla.esac.esa.int/pla/aio/product-action?SIMULATED_MAP.FILE_ID="
      + npipe_filename(100, REAL, "A"))


## 3. Download missing files from PLA

In [ ]:
import time
import urllib.request

PAUSE_S = 0.5
CHUNK = 1 << 20

def npipe_url(freq, real, subset=""):
    base = "https://pla.esac.esa.int/pla/aio/product-action"
    return f"{base}?SIMULATED_MAP.FILE_ID={npipe_filename(freq, real, subset)}"

def download_one(freq, real, subset):
    dest = out_path(freq, real, subset)
    if dest.is_file():
        return "skip"
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + ".part")
    try:
        with urllib.request.urlopen(npipe_url(freq, real, subset)) as r, open(tmp, "wb") as fh:
            if r.status != 200:
                tmp.unlink(missing_ok=True)
                return f"http{r.status}"
            while True:
                buf = r.read(CHUNK)
                if not buf:
                    break
                fh.write(buf)
        if tmp.stat().st_size < 1_000_000:
            tmp.unlink(missing_ok=True)
            return "err:tiny_response"
        tmp.rename(dest)
        inventory[(freq, subset)] = dest
        return f"ok ({dest.stat().st_size/1e6:.0f} MB)"
    except Exception as e:  # noqa: BLE001
        tmp.unlink(missing_ok=True)
        return f"err:{e}"

if not DO_DOWNLOAD:
    print("DO_DOWNLOAD=False — skipping PLA.")
elif not missing:
    print("Everything already on disk — nothing to download.")
else:
    print(f"Downloading {len(missing)} file(s) from PLA...")
    for freq, subset in missing:
        status = download_one(freq, REAL, subset)
        print(f"  {freq}GHz {subset} mc_{REAL:05d}: {status}", flush=True)
        if status.startswith("ok"):
            time.sleep(PAUSE_S)


## 4. Visualise

In [ ]:
import healpy as hp
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

mpl.rcParams.update({
    "text.usetex": False, "font.family": "serif",
    "axes.labelsize": 11, "axes.titlesize": 11, "figure.dpi": 120,
})

for freq in FREQUENCIES:
    for subset in SPLITS:
        inventory[(freq, subset)] = find_local(freq, REAL, subset)

def load_T(path):
    return hp.read_map(str(path), field=0, dtype=np.float64)

gallery = []
for freq in FREQUENCIES:
    p = inventory.get((freq, "A"))
    if p is not None:
        gallery.append((freq, p))

print(f"{len(gallery)} map(s) available to plot")
for freq, p in gallery:
    print(f"  {freq}GHz <- {p}")


In [ ]:
if not gallery:
    raise RuntimeError("No noise maps under OUT_ROOT. Re-run the download cell.")

n = len(gallery)
ncols = min(3, n)
nrows = int(np.ceil(n / ncols))
fig = plt.figure(figsize=(5.2 * ncols, 3.6 * nrows))
for i, (freq, path) in enumerate(gallery):
    m = load_T(path)
    vmin, vmax = np.percentile(m, [1, 99])
    unit = "K_CMB" if freq <= 353 else "MJy/sr"
    hp.mollview(
        m, fig=fig.number, sub=(nrows, ncols, i + 1),
        title=f"{freq} GHz\n{path.name}",
        unit=unit, min=vmin, max=vmax, cmap="RdBu_r", notext=True,
    )
fig.suptitle(f"NPIPE noise A-split  mc_{REAL:05d}", y=1.01, fontsize=13)
out = FIG_DIR / f"planck_noise_gallery_mc{REAL:05d}.png"
fig.savefig(out, bbox_inches="tight", dpi=150)
print("wrote", out)
plt.show()


### Local noise depth

In [ ]:
freq, path = gallery[0]
m = load_T(path)
nside_lo = 16
m_nest = hp.reorder(m, r2n=True)
sub = m_nest.reshape(hp.nside2npix(nside_lo), -1)
rms = sub.std(axis=1)
fig = plt.figure(figsize=(8, 4.5))
hp.mollview(
    rms, nest=True, fig=fig.number,
    title=f"Local RMS  {freq} GHz A  mc_{REAL:05d}",
    unit="RMS", cmap="magma",
    min=np.percentile(rms, 5), max=np.percentile(rms, 95),
)
out = FIG_DIR / f"planck_noise_depth_{freq}GHz_mc{REAL:05d}.png"
fig.savefig(out, bbox_inches="tight", dpi=150)
print(f"{freq} GHz: min={rms.min():.3e} median={np.median(rms):.3e} "
      f"max={rms.max():.3e} max/min={rms.max()/rms.min():.1f}")
print("wrote", out)
plt.show()


### A vs B residual

In [ ]:
pair = None
for freq in FREQUENCIES:
    pa, pb = inventory.get((freq, "A")), inventory.get((freq, "B"))
    if pa is not None and pb is not None:
        pair = (freq, pa, pb)
        break
if pair is None:
    print("No A/B pair on disk yet.")
else:
    freq, pa, pb = pair
    a, b = load_T(pa), load_T(pb)
    diff = a - b
    vmax = np.percentile(np.abs(diff), 99)
    fig = plt.figure(figsize=(14, 4))
    for i, (m, title) in enumerate([
        (a, f"{freq} GHz A"), (b, f"{freq} GHz B"), (diff, f"{freq} GHz A−B"),
    ], 1):
        lo, hi = ((-vmax, vmax) if i == 3 else np.percentile(m, [1, 99]))
        hp.mollview(m, fig=fig.number, sub=(1, 3, i), title=title,
                    cmap="RdBu_r", min=lo, max=hi, notext=True)
    out = FIG_DIR / f"planck_noise_AB_{freq}GHz_mc{REAL:05d}.png"
    fig.savefig(out, bbox_inches="tight", dpi=150)
    print(f"A std={a.std():.3e} B std={b.std():.3e} "
          f"A−B std={diff.std():.3e} corr={np.corrcoef(a,b)[0,1]:.4f}")
    print("wrote", out)
    plt.show()
